# MedVision Dual-Modality Training (Kaggle T4x2)
This notebook fine-tunes Bio_ClinicalBERT and TrOCR in isolated streams using HuggingFace `Trainer` and `accelerate` for dual-GPU utilization.

In [ ]:
!pip install transformers accelerate evaluate jiwer datasets pillow

## 1. Setup & Imports

In [ ]:
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    Trainer, 
    TrainingArguments,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator
)
from PIL import Image
import evaluate
import numpy as np

# Verify Multi-GPU
print(f"GPUs available: {torch.cuda.device_count()}")
os.environ["WANDB_DISABLED"] = "true" # Disable wandb for clean kaggle runs


## 2. Bio_ClinicalBERT Specialty Classification

In [ ]:
# Load Text Data
bert_data_path = "/kaggle/input/datasets/abhigtm19/medvision-nlp/dataset/processed/bio_clinicalbert_dataset.jsonl"
if os.path.exists(bert_data_path):
    df_bert = pd.read_json(bert_data_path, lines=True)
    
    # Encode labels
    labels = df_bert['label'].unique().tolist()
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for i, l in enumerate(labels)}
    
    df_bert['label_id'] = df_bert['label'].map(label2id)
    dataset_bert = Dataset.from_pandas(df_bert)
    dataset_bert = dataset_bert.train_test_split(test_size=0.1, seed=42)
    
    tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)
        
    tokenized_datasets = dataset_bert.map(tokenize_function, batched=True)
    tokenized_datasets = tokenized_datasets.remove_columns(['id', 'source', 'modality', 'text', 'label'])
    tokenized_datasets = tokenized_datasets.rename_column("label_id", "labels")
    tokenized_datasets.set_format("torch")
    
    model_bert = AutoModelForSequenceClassification.from_pretrained(
        "emilyalsentzer/Bio_ClinicalBERT", 
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id
    )
    
    metric = evaluate.load("f1")
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels, average="macro")
        
    training_args = TrainingArguments(
        output_dir="/kaggle/working/bert-results",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        fp16=True, # T4 Mixed Precision
        load_best_model_at_end=True,
    )
    
    # Calculate Class Weights to penalize majority classes
    import torch.nn as nn
    class_counts = df_bert['label_id'].value_counts().sort_index().values
    total_samples = len(df_bert)
    num_classes = len(labels)
    class_weights = torch.tensor([total_samples / (num_classes * max(c, 1)) for c in class_counts], dtype=torch.float)
    
    class ImbalancedTrainer(Trainer):
        def __init__(self, *args, class_weights=None, **kwargs):
            super().__init__(*args, **kwargs)
            if class_weights is not None:
                self.class_weights = class_weights.to(self.args.device)
            else:
                self.class_weights = None

        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            logits = outputs.logits
            if self.class_weights is not None:
                loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
            else:
                loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss
    
    trainer = ImbalancedTrainer(
        model=model_bert,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )
    
    print("Starting Bio_ClinicalBERT Training with Weighted Loss...")
    trainer.train()
    trainer.save_model("/kaggle/working/best_bio_clinicalbert")
else:
    print(f"Dataset not found at {bert_data_path}")


## 3. TrOCR Handwritten Prescription Fine-Tuning

In [ ]:
# Load Vision Data
trocr_data_path = "/kaggle/input/datasets/abhigtm19/medvision-nlp/trocr_dataset.jsonl"
image_base_dir = "/kaggle/input/datasets/abhigtm19/medvision-nlp/" # Base dir for relative file_paths

if os.path.exists(trocr_data_path):
    df_trocr = pd.read_json(trocr_data_path, lines=True)
    dataset_trocr = Dataset.from_pandas(df_trocr)
    dataset_trocr = dataset_trocr.train_test_split(test_size=0.1, seed=42)
    
    processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
    model_trocr = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
    
    # Configure TrOCR parameters
    model_trocr.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model_trocr.config.pad_token_id = processor.tokenizer.pad_token_id
    model_trocr.config.vocab_size = model_trocr.config.decoder.vocab_size
    model_trocr.config.eos_token_id = processor.tokenizer.sep_token_id
    model_trocr.config.max_length = 128
    model_trocr.config.early_stopping = True
    model_trocr.config.no_repeat_ngram_size = 3
    model_trocr.config.length_penalty = 2.0
    model_trocr.config.num_beams = 4
    
    def process_trocr_batch(examples):
        # Load and process images
        pixel_values = []
        for file_path in examples['file_path']:
            img_path = os.path.join(image_base_dir, file_path)
            image = Image.open(img_path).convert("RGB")
            pixel_values.append(processor(image, return_tensors="pt").pixel_values.squeeze())
            
        # Process text
        labels = processor.tokenizer(
            examples['text'], 
            padding="max_length", 
            max_length=128,
            truncation=True
        ).input_ids
        
        # Replace padding token id with -100 to ignore loss
        labels = [[-100 if token == processor.tokenizer.pad_token_id else token for token in label] for label in labels]
        
        return {"pixel_values": pixel_values, "labels": labels}
        
    encoded_trocr = dataset_trocr.map(process_trocr_batch, batched=True, remove_columns=dataset_trocr["train"].column_names)
    encoded_trocr.set_format(type="torch")
    
    cer_metric = evaluate.load("cer")
    
    def compute_cer(pred):
        labels_ids = pred.label_ids
        pred_ids = pred.predictions
        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
        labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
        label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
        cer = cer_metric.compute(predictions=pred_str, references=label_str)
        return {"cer": cer}
        
    training_args_trocr = Seq2SeqTrainingArguments(
        output_dir="/kaggle/working/trocr-results",
        predict_with_generate=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        per_device_train_batch_size=8, # TrOCR is heavy, use smaller batch
        per_device_eval_batch_size=8,
        fp16=True, # T4 Mixed Precision
        learning_rate=4e-5,
        num_train_epochs=5,
        load_best_model_at_end=True,
    )
    
    trainer_trocr = Seq2SeqTrainer(
        model=model_trocr,
        tokenizer=processor.feature_extractor,
        args=training_args_trocr,
        compute_metrics=compute_cer,
        train_dataset=encoded_trocr["train"],
        eval_dataset=encoded_trocr["test"],
        data_collator=default_data_collator,
    )
    
    print("Starting TrOCR Fine-Tuning...")
    trainer_trocr.train()
    trainer_trocr.save_model("/kaggle/working/best_trocr_model")
    processor.save_pretrained("/kaggle/working/best_trocr_model")
else:
    print(f"Dataset not found at {trocr_data_path}")
